In [0]:
pip install kagglehub[pandas-datasets]

In [0]:
import os
os.environ["KAGGLE_USERNAME"] = dbutils.secrets.get("nba_insights", "kaggle_username")
os.environ["KAGGLE_KEY"] = dbutils.secrets.get("nba_insights", "kaggle_key")

In [0]:
# Install dependencies as needed:

import kagglehub
from kagglehub import KaggleDatasetAdapter


def get_kaggle_data(file_path):
  return kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "wyattowalsh/basketball",
    file_path,
    # Provide any additional arguments like 
    # sql_query or pandas_kwargs. See the 
    # documenation for more information:
    # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
  )



In [0]:
catalog_name = "nba_insights"
schema_name = "bronze"
games_table = "games"
common_player_info_table = "common_player_info"
inactive_players_table = "inactive_players"

## Ingest game

## Ingest Common Player Info

## Ingest Inactive Players

In [0]:
from pyspark.sql.functions import current_timestamp

def ingest_kaggle_table(spark, get_kaggle_data_fn, csv_path, catalog_name, schema_name, table_name):
    """Reads a Kaggle CSV into a Spark DataFrame, adds an ingestion timestamp,
    and writes it as a managed table (overwrite)."""
    pdf = get_kaggle_data_fn(csv_path)
    df = spark.createDataFrame(pdf)
    df = df.withColumn("ingestion_timestamp", current_timestamp())
    df.write.mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.{table_name}")
    return df


## Ingest game

In [0]:
ingest_kaggle_table(spark, get_kaggle_data, "csv/game.csv", catalog_name, schema_name, games_table)

## Ingest Common Players

In [0]:
# Cell 2
ingest_kaggle_table(spark, get_kaggle_data, "csv/common_player_info.csv", catalog_name, schema_name, common_player_info_table)


## Ingest Inactive Players

In [0]:
ingest_kaggle_table(spark, get_kaggle_data, "csv/inactive_players.csv", catalog_name, schema_name, inactive_players_table)